In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, PolynomialFeatures

Load data

In [ ]:
df = pd.read_csv("nyc-rolling-sales.csv")

In [ ]:
print(df.head())

   Unnamed: 0  BOROUGH   NEIGHBORHOOD  \
0           4        1  ALPHABET CITY   
1           7        1  ALPHABET CITY   
2           8        1  ALPHABET CITY   
3          10        1  ALPHABET CITY   
4          13        1  ALPHABET CITY   

                       BUILDING CLASS CATEGORY TAX CLASS AT PRESENT  BLOCK  \
0  07 RENTALS - WALKUP APARTMENTS                                2A    392   
1  07 RENTALS - WALKUP APARTMENTS                                2B    402   
2  07 RENTALS - WALKUP APARTMENTS                                2A    404   
3  07 RENTALS - WALKUP APARTMENTS                                2B    406   
4  08 RENTALS - ELEVATOR APARTMENTS                               2    387   

   LOT EASE-MENT BUILDING CLASS AT PRESENT                 ADDRESS  ...  \
0    6                                  C2            153 AVENUE B  ...   
1   21                                  C4     154 EAST 7TH STREET  ...   
2   55                                  C2  301 EAST 10TH  

Clean numeric fields

In [ ]:
df['SALE PRICE'] = pd.to_numeric(df['SALE PRICE'], errors='coerce')
df['LAND SQUARE FEET'] = pd.to_numeric(df['LAND SQUARE FEET'].astype(str).str.replace(',', ''), errors='coerce')
df['GROSS SQUARE FEET'] = pd.to_numeric(df['GROSS SQUARE FEET'].astype(str).str.replace(',', ''), errors='coerce')
df['SALE DATE'] = pd.to_datetime(df['SALE DATE'], errors='coerce')




Column Data Types After Cleaning:

SALE PRICE                  float64
LAND SQUARE FEET            float64
GROSS SQUARE FEET           float64
SALE DATE            datetime64[ns]
dtype: object

Sample Rows After Cleaning:

   SALE PRICE  LAND SQUARE FEET  GROSS SQUARE FEET  SALE DATE
0   6625000.0            1633.0             6440.0 2017-07-19
1         NaN            4616.0            18690.0 2016-12-14
2         NaN            2212.0             7803.0 2016-12-09
3   3936272.0            2272.0             6794.0 2016-09-23
4   8000000.0            2369.0             4615.0 2016-11-17
5         NaN            2581.0             9730.0 2017-07-20
6   3192840.0            1750.0             4226.0 2016-09-23
7         NaN            5163.0            21007.0 2017-07-20
8         NaN            1534.0             9198.0 2017-06-20
9  16232000.0            4489.0            18523.0 2016-11-07


Print updated dtypes to verify conversions

In [ ]:
print("Column Data Types After Cleaning:\n")
print(df[['SALE PRICE', 'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'SALE DATE']].dtypes)



Print a few sample rows to see the results

In [ ]:
print("\nSample Rows After Cleaning:\n")
print(df[['SALE PRICE', 'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'SALE DATE']].head(10))

Filter valid data

In [ ]:
df = df[df['SALE PRICE'].notna() & (df['SALE PRICE'] > 0)]
df = df.dropna(subset=['LAND SQUARE FEET', 'GROSS SQUARE FEET']).reset_index(drop=True)

Feature Selection via correlation

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number])
correlations = numeric_cols.corr()['SALE PRICE'].sort_values(ascending=False)
selected_features = correlations[1:6].index.tolist()  # Exclude SALE PRICE itself

Log Transformation

In [ ]:
df['LOG SALE PRICE'] = np.log1p(df['SALE PRICE'])

Polynomial Features

In [ ]:
df['BUILDING AGE'] = 2024 - df['YEAR BUILT']
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[['BUILDING AGE']])
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(['BUILDING AGE']))
df = pd.concat([df, poly_df], axis=1)

Feature Creation

In [ ]:
df['PRICE PER UNIT'] = df['SALE PRICE'] / df['TOTAL UNITS'].replace(0, np.nan)
df['PRICE PER SQFT'] = df['SALE PRICE'] / df['GROSS SQUARE FEET'].replace(0, np.nan)
df['HAS_COMMERCIAL'] = (df['COMMERCIAL UNITS'] > 0).astype(int)

Feature Extraction from ADDRESS

In [ ]:
df['ADDRESS_LEN'] = df['ADDRESS'].str.len()

Label Encoding for BOROUGH

In [ ]:
le = LabelEncoder()
df['BOROUGH_ENC'] = le.fit_transform(df['BOROUGH'])

One-hot Encoding for top 5 NEIGHBORHOODs

In [ ]:
top_5 = df['NEIGHBORHOOD'].value_counts().nlargest(5).index
df['NEIGHBORHOOD_TOP5'] = df['NEIGHBORHOOD'].apply(lambda x: x if x in top_5 else 'Other')
df = pd.get_dummies(df, columns=['NEIGHBORHOOD_TOP5'], prefix='NEIGHBORHOOD')

Standardization and Normalization

In [ ]:
scaler_std = StandardScaler()
scaler_norm = MinMaxScaler()

df['SALE PRICE STD'] = scaler_std.fit_transform(df[['SALE PRICE']])
df['SALE PRICE NORM'] = scaler_norm.fit_transform(df[['SALE PRICE']])

 Preview

In [ ]:
print(df[[
    'SALE PRICE', 'LOG SALE PRICE', 'BUILDING AGE',
    'PRICE PER UNIT', 'PRICE PER SQFT', 'HAS_COMMERCIAL',
    'ADDRESS_LEN', 'BOROUGH_ENC', 'SALE PRICE STD', 'SALE PRICE NORM'
] + [col for col in df.columns if col.startswith('NEIGHBORHOOD_')]].head())


   SALE PRICE  LOG SALE PRICE  BUILDING AGE  BUILDING AGE  PRICE PER UNIT  \
0   6625000.0       15.706361           124         124.0    1.325000e+06   
1   3936272.0       15.185745           111         111.0    3.936272e+05   
2   8000000.0       15.894952           124         124.0    1.333333e+06   
3   3192840.0       14.976422           104         104.0    3.991050e+05   
4  16232000.0       16.602495           104         104.0    6.763333e+05   

   PRICE PER SQFT  HAS_COMMERCIAL  ADDRESS_LEN  BOROUGH_ENC  SALE PRICE STD  \
0     1028.726708               0           12            0        0.342233   
1      579.374742               0           19            0        0.163955   
2     1733.477790               0           22            0        0.433404   
3      755.522953               0           12            0        0.114661   
4      876.315932               0           19            0        0.979234   

   SALE PRICE NORM  NEIGHBORHOOD_BEDFORD STUYVESANT  \
0      